# Stage 23 — TENTACLES native classifier warning triage

This notebook is an **executed audit of the saved per-fold warning counts**, not a certification that all ten native fitted models are healthy. The native R `.rds` files are stored in the user's local WSL environment and are **not** in this Drive package. The separate `r/tentacles_readonly_native_quality_audit.R` + `scripts/run_tentacles_readonly_quality_audit.sh` inspect those objects without retraining; their reports must be reviewed before declaring native model health.

In [ ]:
from pathlib import Path
import os,sys,json
from datetime import datetime,timezone
import pandas as pd
ROOT=Path(os.environ.get('CMPS355_ROOT',Path.cwd())).resolve()
if not (ROOT/'runs/tentacles/summary/native_classifier_warning_audit.csv').is_file():
    raise FileNotFoundError('Missing native TENTACLES warning receipt; connect private Drive runs first')
sys.path.insert(0,str(ROOT/'src'))
RUN_ID=os.environ.get('CMPS355_RUN_ID') or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR=ROOT/'runs/native_warning_triage'/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=False)


In [ ]:
from cmps355.native_warning_triage import report_from_summary
source=ROOT/'runs/tentacles/summary/native_classifier_warning_audit.csv'
warn=report_from_summary(source)
warn.to_csv(RUN_DIR/'warning_categories.csv',index=False)
print('TENTACLES_ORIGINAL_WARNING_COUNTS:')
display(warn)
print('MLP messages:',warn.mlp_invalid_weight_tuning_messages.sum())
print('LightGBM invalid fraction:',warn.lightgbm_invalid_fraction_tuning_messages.sum())
print('Boruta missing importances:',warn.boruta_missing_importance_warnings.sum())

In [ ]:
report={'status':'PASS_LOG_WARNING_TRIAGE_ONLY; NATIVE_MODEL_QUALITY_NOT_YET_CERTIFIED',
  'read_RDS_independently':False,
  'source_path':'runs/tentacles/summary/native_classifier_warning_audit.csv',
  'folds':4,
  'meaning':'Log warning counts do not establish final-fit failure or success; check native model importances and CV metrics in RDS.'}
(RUN_DIR/'report.json').write_text(json.dumps(report,indent=2))
print(report['status']);print('Private report:',RUN_DIR)

## Decision framework

- `mlp_weight_errors`: invalid configurations can arise during tuning; the presence of a fitted model entry does not prove every candidate succeeded. Verify final selected candidate, class probabilities and importance.
- `lightgbm_feature_fraction_errors`: D1 has invalid parameter notices. Verify that **successful** candidates contribute to the final fit; do not treat a failed candidate as missing model evidence without inspecting RDS.
- `boruta_nan_warnings`: nonfinite importance during repeated selection may affect feature sets. Verify whether any final model importance or consensus vote included NAs; inspect the 9-voter consensus matrix.
- `getConsensus` voters: 9 non-XGBoost models with `n.min=5` in the completed run. A consensus vote-list existing is not equivalent to a full model-quality validation.

**Do not rerun all four native fits unless the read-only RDS audit identifies a material failure.**